# 📚 Book → Audiobook (Google Colab)

Run the cells **top to bottom**. Everything (project, books, progress, MP3s) lives in your Google Drive folder
`MyDrive/book_to_audio`, so if Colab disconnects you can re-run and it **resumes where it stopped**.

**First time only:** you'll be asked to upload `book_to_audio_complete.zip` in step 2.

## 1. Mount Google Drive

In [1]:
import os
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/book_to_audio'
BOOKS_DIR = f'{PROJECT_DIR}/books'
OUTPUT_DIR = f'{PROJECT_DIR}/output'
for d in (PROJECT_DIR, BOOKS_DIR, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)
print('Project folder:', PROJECT_DIR)

Mounted at /content/drive
Project folder: /content/drive/MyDrive/book_to_audio


## 2. Install the project and dependencies

In [2]:
import os, shutil, subprocess, sys, zipfile
from google.colab import files

UPDATE_PROJECT = False  #@param {type:"boolean"}
# ^ tick this when you have a newer book_to_audio_complete.zip to install over the Drive copy

if UPDATE_PROJECT or not os.path.exists(f'{PROJECT_DIR}/book_to_audio.py'):
    print('Upload book_to_audio_complete.zip ...')
    uploaded = files.upload()
    zips = [n for n in uploaded if n.lower().endswith('.zip')]
    if not zips:
        raise SystemExit('No .zip uploaded. Re-run this cell and choose book_to_audio_complete.zip')
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall('/content/_unzipped')
    src = '/content/_unzipped/book_to_audio'
    shutil.copytree(src, PROJECT_DIR, dirs_exist_ok=True)
    print('Project installed to Drive.')
else:
    print('Project already in Drive (tick UPDATE_PROJECT to replace it).')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', '-r', f'{PROJECT_DIR}/requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'edge-tts'], check=True)  # old versions get HTTP 403
print('ffmpeg:', shutil.which('ffmpeg') or 'NOT FOUND')

Project already in Drive (tick UPDATE_PROJECT to replace it).
ffmpeg: /usr/bin/ffmpeg


## 2b. Check that Edge-TTS works from this Colab
Microsoft sometimes blocks cloud IPs (HTTP 403). **Skip this cell if you use `ENGINE = "kokoro"`** (step 3).

In [3]:
import glob, os, subprocess, sys

shutil.rmtree('/content/_sample', ignore_errors=True)
r = subprocess.run([sys.executable, 'book_to_audio.py', '-o', '/content/_sample', '--sample', 'en-US-AvaMultilingualNeural'],
                   cwd=PROJECT_DIR, capture_output=True, text=True)
print(r.stderr[-600:])
ok = any(os.path.getsize(f) > 0 for f in glob.glob('/content/_sample/*.mp3'))
if ok:
    print('\u2705 Edge-TTS works from this Colab. Continue to step 3.')
else:
    print('\u274c Edge-TTS is being rejected from this Colab (usually an IP block / HTTP 403).')
    print('   Try Runtime -> Disconnect and delete runtime, then reconnect (new IP) and re-run steps 1-2b.')
    print('   If it keeps failing, run the project on your own computer instead (see README).')

Synthesizing 1 sample(s) into /content/_sample/ ...
  -> /content/_sample/sample_en-US-AvaMultilingualNeural.mp3

✅ Edge-TTS works from this Colab. Continue to step 3.


## 3. Settings

Put your `.epub` / `.pdf` in `MyDrive/book_to_audio/books/` (or leave it missing and you'll be prompted to upload).

**MODE options**
**ENGINE:** `kokoro` (offline, recommended on Colab) or `edge` (Microsoft online).

- `single_voice` – one narrator, quick
- `multi_voice` – regex heuristic, a voice per character
- `multi_voice_whole_book` – smarter global pass
- `multi_voice_light_model` – small Hugging Face model (GPU recommended)
- `multi_voice_heavy_model` – local LLM via Ollama (GPU runtime required)
- `analyze_only` – just show the detected cast, make no audio
- `list_chapters` – just list chapters

In [4]:
BOOK_FILE = "Secret Garden Frances Hodgson Burnett.epub"  #@param {type:"string"}
ENGINE = "kokoro"  #@param ["kokoro", "edge"]
# ^ kokoro = offline voices that work on Colab (model downloads once into Drive). edge = Microsoft online voices (blocked on Colab for you).
KOKORO_MODEL = "full quality"  #@param ["auto", "fast (int8, CPU)", "full quality"]
# ^ auto = fast int8 model on a CPU runtime, full quality on a GPU runtime. int8 is quicker but a little lower quality.
MODE = "multi_voice_heavy_model"  #@param ["single_voice", "multi_voice", "multi_voice_whole_book", "multi_voice_light_model", "multi_voice_heavy_model", "analyze_only", "list_chapters"]
CHAPTERS = ""  #@param {type:"string"}
# ^ e.g. 1-2 to test, or 1,3,5-8. Leave blank for the whole book.
PER_CHAPTER = False  #@param {type:"boolean"}
NARRATOR_GENDER = "none"  #@param ["none", "male", "female"]
VOICE = ""  #@param {type:"string"}
# ^ narrator voice. Blank = automatic. kokoro: af_heart, am_michael, bf_emma ...  edge: en-US-AriaNeural ...
RATE = "+0%"  #@param {type:"string"}
VOICE_PROFILE = "novel"  #@param ["novel", "news", "conversation"]
MAX_CHARACTERS = 30  #@param {type:"integer"}
LLM_MODEL = "llama3.1:8b"  #@param ["qwen3:14b", "phi4", "gemma3:12b", "mistral-nemo", "llama3.1:8b"] {allow-input: true}
# ^ only used by multi_voice_heavy_model. 12-14B models fit a free T4 GPU; llama3.1:8b is the lighter fallback.
FRESH_START = False  #@param {type:"boolean"}

## 4. (Heavy mode only) Start Ollama
This cell does nothing unless `MODE` is `multi_voice_heavy_model`. Use *Runtime → Change runtime type → GPU* first.

In [5]:
import time, urllib.request

def ollama_up():
    try:
        urllib.request.urlopen('http://localhost:11434', timeout=2)
        return True
    except Exception:
        return False

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-1500:]

if MODE == "multi_voice_heavy_model":
    has_gpu = bool(shutil.which('nvidia-smi')) and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
    if not has_gpu:
        print('WARNING: no GPU runtime. An 8B model on the CPU is far too slow for a whole book.')
        print('Use MODE = multi_voice_whole_book, or Runtime -> Change runtime type -> T4 GPU.\n')

    if not shutil.which('ollama'):
        # The installer needs zstd (missing on Colab) and lspci (GPU detection)
        sh('apt-get -qq update && apt-get -qq install -y zstd pciutils')
        code_, out = sh('curl -fsSL https://ollama.com/install.sh | sh')
        if not shutil.which('ollama'):
            print('Installer failed, trying a manual install ...\n', out)
            code_, out = sh('curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst | zstd -d | tar -xf - -C /usr')
        if not shutil.which('ollama'):
            raise SystemExit('Could not install Ollama:\n' + out)

    if not ollama_up():
        subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(60):
            if ollama_up():
                break
            time.sleep(1)
    if not ollama_up():
        raise SystemExit('Ollama is installed but did not start. Re-run this cell once; if it repeats, use whole_book mode.')

    subprocess.run(['ollama', 'pull', LLM_MODEL], check=True)
    print('Ollama ready with', LLM_MODEL)
else:
    print('Skipped (not heavy mode).')

Ollama ready with qwen3:14b


## 5. Run

In [6]:
import os, subprocess, sys

# Locate the book (upload it if it isn't in Drive yet)
book_path = os.path.join(BOOKS_DIR, BOOK_FILE)
if not os.path.exists(book_path):
    print(f'{BOOK_FILE} not found in {BOOKS_DIR}. Upload it now:')
    up = files.upload()
    for name, data in up.items():
        with open(os.path.join(BOOKS_DIR, name), 'wb') as f:
            f.write(data)
    BOOK_FILE = list(up)[0]
    book_path = os.path.join(BOOKS_DIR, BOOK_FILE)

stem = os.path.splitext(BOOK_FILE)[0]
out_path = os.path.join(OUTPUT_DIR, stem if PER_CHAPTER else f'{stem}.mp3')

gpu = False
if ENGINE == 'kokoro' and MODE not in ('list_chapters', 'analyze_only'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'kokoro-onnx'], check=True)
    subprocess.run('apt-get -qq install -y espeak-ng > /dev/null 2>&1', shell=True)  # best effort
    os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)  # model files (~340 MB) are cached in Drive
    gpu = shutil.which('nvidia-smi') and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
    if gpu:
        print('GPU runtime detected: switching to onnxruntime-gpu for much faster synthesis ...')
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'onnxruntime'], capture_output=True)
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'onnxruntime-gpu'])
        if r.returncode != 0:  # never leave the runtime without onnxruntime
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime'])
    else:
        print('No GPU runtime: Kokoro will use the CPU, which is slow for a whole book.')
        print('Tip: Runtime -> Change runtime type -> T4 GPU, then re-run from step 1.')

cmd = [sys.executable, 'book_to_audio.py', book_path, '--voice-profile', VOICE_PROFILE]

if MODE == 'list_chapters':
    cmd += ['--list-chapters']
elif MODE == 'analyze_only':
    cmd += ['--analyze-only', '--multi-voice', '--max-characters', str(MAX_CHARACTERS)]
else:
    cmd += ['-o', out_path, '--rate', RATE, '--engine', ENGINE]
    if ENGINE == 'kokoro':
        fast = KOKORO_MODEL.startswith('fast') or (KOKORO_MODEL == 'auto' and not gpu)
        model_file = 'kokoro-v1.0.int8.onnx' if fast else 'kokoro-v1.0.onnx'
        print('Kokoro model:', 'fast (int8)' if fast else 'full quality', '| GPU runtime:', bool(gpu))
        cmd += ['--kokoro-model', f'{PROJECT_DIR}/models/{model_file}',
                '--kokoro-voices', f'{PROJECT_DIR}/models/voices-v1.0.bin']
        if fast:
            cmd += ['--kokoro-fast']
    if MODE != 'single_voice':
        cmd += ['--multi-voice', '--max-characters', str(MAX_CHARACTERS)]
    if PER_CHAPTER:
        cmd += ['--per-chapter']
    if CHAPTERS.strip():
        cmd += ['--chapters', CHAPTERS.strip()]
    if FRESH_START:
        cmd += ['--fresh']

if MODE == 'multi_voice_whole_book':
    cmd += ['--whole-book-context']
elif MODE == 'multi_voice_light_model':
    cmd += ['--light-model']
elif MODE == 'multi_voice_heavy_model':
    cmd += ['--heavy-model', '--llm-model', LLM_MODEL]

if VOICE.strip():
    cmd += ['--voice', VOICE.strip()]
if NARRATOR_GENDER != 'none':
    cmd += ['--narrator-gender', NARRATOR_GENDER]

print('Running:', ' '.join(cmd), '\n')
proc = subprocess.Popen(cmd, cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    proc.wait()
    print('\nStopped. Progress is saved - run this cell again to resume.')
print('\nExit code:', proc.returncode)
if proc.returncode == 0 and MODE not in ('list_chapters', 'analyze_only'):
    print('Output:', out_path)

GPU runtime detected: switching to onnxruntime-gpu for much faster synthesis ...
Kokoro model: full quality | GPU runtime: True
Running: /usr/bin/python3 book_to_audio.py /content/drive/MyDrive/book_to_audio/books/Secret Garden Frances Hodgson Burnett.epub --voice-profile novel -o /content/drive/MyDrive/book_to_audio/output/Secret Garden Frances Hodgson Burnett.mp3 --rate +0% --engine kokoro --kokoro-model /content/drive/MyDrive/book_to_audio/models/kokoro-v1.0.onnx --kokoro-voices /content/drive/MyDrive/book_to_audio/models/voices-v1.0.bin --multi-voice --max-characters 30 --heavy-model --llm-model qwen3:14b 

Found saved progress in /content/drive/MyDrive/book_to_audio/output/.Secret Garden Frances Hodgson Burnett.book_to_audio_progress - resuming.
Using cached cast analysis: /content/drive/MyDrive/book_to_audio/books/.Secret Garden Frances Hodgson Burnett.cast.heavy.json
'Secret Garden' - 22 named character(s) found, 2262 dialogue line(s) total
  [     c1] Mary Lennox               

## 6. Download the result (optional)
The MP3 is already saved in `MyDrive/book_to_audio/output/`. For large books, downloading from Drive is faster than from here.

In [7]:
import glob
from google.colab import files

if PER_CHAPTER:
    archive = shutil.make_archive(f'/content/{stem}_chapters', 'zip', out_path)
    files.download(archive)
elif os.path.exists(out_path):
    files.download(out_path)
else:
    print('No output file found yet:', out_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>